In [ ]:
import sys
import os

# Add the repository root to sys.path
sys.path.append(os.path.abspath(".."))

import numpy as np
from utils.config import load_config, Configuration
import pandas as pd
from utils.utils import retrieve_stacked_betas, retrieve_stacked_betas_test, retrieve_roi_mask, filter_roi_mask

config = load_config("config.yaml")

In [ ]:

subj = 1
mask_value = 1
set_to_take = "subj_01"



mask = retrieve_roi_mask(config, subj, set_to_take)
betas, _ = retrieve_stacked_betas(config, subj, "averaged", 0, subj_to_check=set_to_take)
betas_test = retrieve_stacked_betas_test(config, subj, subj_to_check=set_to_take)


mask_voxel_indices = filter_roi_mask(mask_value, mask)
mask_voxel_indices = mask_voxel_indices[0]

betas_masked = betas.T[mask_voxel_indices]
betas_test_masked = betas_test.T[mask_voxel_indices]



data = np.load(f"data/mds_dir/{set_to_take}/subj_{subj:02d}/mask_{mask_value}_averaged_mds.npy")
metadata = np.load(f"data/rdm_dir/{set_to_take}/subj_{subj:02d}/metadata.npy")

fitted_voxels_df = pd.read_excel(f"data/gaussian_results/{set_to_take}/subj{subj:02d}/fitted_voxels_mask_{mask_value}.xlsx", index_col=None)
# fitted_voxels_df = fitted_voxels_df.sort_values(by='sigma')


In [ ]:
fitted_voxels_df

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# Plot the results
plt.scatter(data[:, 0], data[:, 1], s=2)
plt.title("MDS Plot of Neural Response Similarity")
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.show()

In [ ]:
import os
from PIL import Image
images = [Image.open(os.path.join(config.images_target_dir, f"{i}.jpg")) for i in metadata]
titles = [f"{i}" for i in metadata]


In [ ]:

# Create and run the app
images = [os.path.join(config.images_target_dir, f"{i}.jpg") for i in metadata]


In [ ]:
from dash import Dash, html, dcc
import plotly.graph_objects as go
from dash.dependencies import Input, Output, State
from dash import callback_context  # Import callback context

from PIL import Image
import base64
import io
import json
import os
import numpy as np



def create_interactive_image_scatter_dash(data, images, titles=None, width=1500, height=1500):
    """
    Create an interactive scatter plot using Dash where images are displayed at coordinates.
    
    Parameters:
    data: numpy array of shape (n_samples, 2) containing x,y coordinates
    images: list of PIL Images or paths to image files
    titles: optional list of titles for each point
    """

    subj_to_pick = "shared"

    face_detection_data_path = os.path.join(config.excel_files_target_dir, subj_to_pick, config.face_detection_results_path)



    # Load face detection results
    with open(face_detection_data_path, "r") as f:
        face_detection_data = json.load(f)

    image_strings = []
    thumbnail_size = (200, 200)  # Size of images on the plot

    for img in images:
        file_basename = os.path.basename(img)
        detection_bbox = [e for e in face_detection_data if e["file_name"] == file_basename][0]["detection"][0]
        if isinstance(img, str):  # If it's a path
            img = Image.open(img)
        img = img.crop((detection_bbox[0], detection_bbox[1], detection_bbox[2], detection_bbox[3]))
        
        # Create thumbnails
        img_thumb = img.copy()
        img_thumb.thumbnail(thumbnail_size)
        img_full = img.resize((400, 400))
        
        # Convert images to base64
        thumb_buffered = io.BytesIO()
        img_thumb.save(thumb_buffered, format="PNG")
        thumb_str = base64.b64encode(thumb_buffered.getvalue()).decode()

        full_buffered = io.BytesIO()
        img_full.save(full_buffered, format="PNG")
        full_str = base64.b64encode(full_buffered.getvalue()).decode()

        image_strings.append({'thumbnail': thumb_str, 'full': full_str})

    # Create Dash app
    app = Dash(__name__)

    # Define heatmap grid
    grid_size = 500
    x_range = np.linspace(-1, 1, grid_size)
    y_range = np.linspace(-1, 1, grid_size)
    X, Y = np.meshgrid(x_range, y_range)

    # Function to generate Gaussian heatmap
    def get_gaussian_heatmap(x0, y0, sigma):
        Z = (1 / (2 * np.pi * sigma**2)) * np.exp(-((X - x0)**2 + (Y - y0)**2) / (2 * sigma**2))
        return Z / np.max(Z)  # Normalize

    # Initialize figure
    fig = go.Figure()

    # Add initial heatmap
    
    row = fitted_voxels_df.iloc[0]
        
    voxel_i = row.name
    x0, y0, sigma = row[['x0', 'y0', 'sigma']]


    Z = get_gaussian_heatmap(x0, y0, sigma)
    fig.add_trace(go.Heatmap(
        z=Z, x=x_range, y=y_range,
        colorscale='Reds',
        opacity=0.4,
        showscale=False
    ))

    # Add invisible scatter points to maintain plot range
    fig.add_trace(go.Scatter(
        x=[-1, 1], y=[-1, 1], mode='markers',
        marker=dict(size=0, opacity=0), hoverinfo='skip'
    ))

    # Add data points as invisible markers
    fig.add_trace(go.Scatter(
        x=data[:, 0], y=data[:, 1], mode='markers',
        marker=dict(size=0, opacity=0), hoverinfo='skip'
    ))

    # Add images as layout shapes
    for i, (coords, img_data) in enumerate(zip(data, image_strings)):
        fig.add_layout_image(dict(
            source=f'data:image/png;base64,{img_data["thumbnail"]}',
            x=coords[0], y=coords[1],
            xref="x", yref="y",
            sizex=0.1, sizey=0.1,
            xanchor="center", yanchor="middle",
            layer="above"
        ))

    # Update layout
    fig.update_layout(
        title="MDS Plot of Neural Response Similarity",
        xaxis_title="Dimension 1",
        yaxis_title="Dimension 2",
        hovermode='closest',
        xaxis=dict(range=[-1, 1], fixedrange=True, constrain='domain'),
        yaxis=dict(range=[-1, 1], fixedrange=True, scaleanchor="x", scaleratio=1, constrain='domain'),
        width=width, height=height, showlegend=False
    )

    # Define app layout with an information panel
    app.layout = html.Div([
        dcc.Graph(id='scatter-plot', figure=fig, config={
            'doubleClick': 'reset',
            'scrollZoom': False,
            'displayModeBar': True,
            'modeBarButtonsToRemove': ['zoom', 'pan', 'select', 'lasso2d']
        }),
        html.Div(id='image-container', style={'textAlign': 'center', 'marginTop': '20px'}),
        html.Div(id='info-panel', style={'textAlign': 'center', 'fontSize': '20px', 'marginBottom': '10px'}),
        html.Div([
            html.Button("Previous", id="prev-btn", n_clicks=0),
            html.Button("Next", id="next-btn", n_clicks=0),
        ], style={'textAlign': 'center', 'marginTop': '20px'}),
        dcc.Store(id='index-store', data=0)
    ])

    # Callback to update heatmap and information panel
    @app.callback(
        [Output('scatter-plot', 'figure'),
         Output('index-store', 'data'),
         Output('info-panel', 'children')],
        [Input('prev-btn', 'n_clicks'),
         Input('next-btn', 'n_clicks')],
        [State('index-store', 'data')]
    )
    def update_distribution(prev_clicks, next_clicks, current_index):
        if current_index is None:
            current_index = 0


        # new_index = (current_index + (next_clicks - prev_clicks)) % len(distributions)
        # print(f"{new_index=}")


        # Identify which button was clicked
        ctx = callback_context
        if not ctx.triggered:
            return fig, current_index, "Click a button to change distribution."

        button_id = ctx.triggered[0]['prop_id'].split('.')[0]  # Get button ID

        # Update index based on the clicked button
        if button_id == "prev-btn":
            new_index = (current_index - 1) % len(fitted_voxels_df.index)
        elif button_id == "next-btn":
            new_index = (current_index + 1) % len(fitted_voxels_df.index)
        else:
            new_index = current_index  # No valid button pressed


        # Get new distribution
        row = fitted_voxels_df.iloc[new_index]
        
        voxel_i = row.name
        x0, y0, sigma = row[['x0', 'y0', 'sigma']]
        Z = get_gaussian_heatmap(x0, y0, sigma)

        # Create a copy of the figure and update heatmap
        updated_fig = go.Figure(fig)
        updated_fig.data[0].z = Z  # Update heatmap only

        info_text = f"Index: {voxel_i} | Mean: ({x0:.2f}, {y0:.2f}) | Sigma: {sigma:.2f}"
        return updated_fig, new_index, info_text

    # Callback to display larger image when clicking on a point
    @app.callback(
        Output('image-container', 'children'),
        Input('scatter-plot', 'clickData')
    )
    def display_image(clickData):
        if clickData is None:
            return html.Div("Click an image to view it larger")

        point_index = clickData['points'][0]['pointIndex']
        img_src = f'data:image/png;base64,{image_strings[point_index]["full"]}'
        title = titles[point_index] if titles else f"Point {point_index}"

        return html.Div([
            html.H4(title),
            html.Img(src=img_src, style={'maxWidth': '400px'})
        ])

    return app

In [ ]:
# app = create_interactive_image_scatter_dash(data, images, width=1000, height=1000)
# app.run_server(debug=True, port=8050)

In [ ]:
from dash import Dash, html, dcc
import plotly.graph_objects as go
from dash.dependencies import Input, Output, State
from dash import callback_context
import json
import os
import numpy as np

def create_interactive_circle_scatter_dash(data, amplitudes, test_amplitudes, titles=None,
                                           scaling_factor=0.1, test_scaling_factor=0.1,
                                           max_radius=0.2,  # Maximum allowed radius for any shape.
                                           width=1500, height=1500):
    """
    Create an interactive scatter plot using Dash where each data point is represented by:
      - A training amplitude shape (blue): rendered as a circle for positive values,
        or as a square for negative values.
      - A test amplitude shape (red): rendered as a circle for positive values,
        or as a square for negative values.
    
    Both the sigma (Gaussian) heatmap and the amplitude shapes update when the index changes.
    
    The info text displays:
      - The current index, mean coordinates, and sigma.
      - The minimum and maximum training amplitudes (blue) and test amplitudes (red).
      - A note on which color represents which set.
    
    Parameters:
      data: numpy array of shape (n_samples, 2) with x,y coordinates.
      amplitudes: 2D array-like, where amplitudes[i] contains amplitude values (training) for each data point at index i.
      test_amplitudes: 2D array-like, where test_amplitudes[i] contains amplitude values (test) for each data point at index i.
      titles: optional list of titles for each data point.
      scaling_factor: factor to scale training amplitudes to shape size.
      test_scaling_factor: factor to scale test amplitudes to shape size.
      max_radius: maximum allowed radius for any shape.
      width, height: dimensions of the figure.
    """
    
    # Load additional data if necessary (e.g., for sigma heatmap).  
    # This line is kept from your original code.

    app = Dash(__name__)

    # Define the heatmap grid.
    grid_size = 500
    x_range = np.linspace(-1, 1, grid_size)
    y_range = np.linspace(-1, 1, grid_size)
    X, Y = np.meshgrid(x_range, y_range)

    def get_gaussian_heatmap(x0, y0, sigma):
        Z = np.exp(-((X - x0)**2 + (Y - y0)**2) / (2 * sigma**2))
        return Z / np.max(Z)

    # Helper function to build amplitude shapes for training and test.
    def get_amplitude_shapes(data, amps, test_amps, scaling_factor, test_scaling_factor, max_radius):
        shapes = []
        for i, coords in enumerate(data):
            # Training amplitude shape (blue)
            amp = amps[i]
            if amp < 0:
                # Use a square for negative amplitudes.
                r = min(abs(amp) * scaling_factor, max_radius)
                shapes.append(dict(
                    type="rect",
                    xref="x", yref="y",
                    x0=coords[0] - r, y0=coords[1] - r,
                    x1=coords[0] + r, y1=coords[1] + r,
                    line=dict(color="blue", width=2),
                    fillcolor="blue",
                    opacity=0.8,
                    layer="above"
                ))
            else:
                r = min(amp * scaling_factor, max_radius)
                shapes.append(dict(
                    type="circle",
                    xref="x", yref="y",
                    x0=coords[0] - r, y0=coords[1] - r,
                    x1=coords[0] + r, y1=coords[1] + r,
                    line=dict(color="blue", width=2),
                    fillcolor="blue",
                    opacity=0.8,
                    layer="above"
                ))
            
            # Test amplitude shape (red)
            test_amp = test_amps[i]
            if test_amp < 0:
                r_test = min(abs(test_amp) * test_scaling_factor, max_radius)
                shapes.append(dict(
                    type="rect",
                    xref="x", yref="y",
                    x0=coords[0] - r_test, y0=coords[1] - r_test,
                    x1=coords[0] + r_test, y1=coords[1] + r_test,
                    line=dict(color="red", width=2),
                    fillcolor="red",
                    opacity=0.4,
                    layer="above"
                ))
            else:
                r_test = min(test_amp * test_scaling_factor, max_radius)
                shapes.append(dict(
                    type="circle",
                    xref="x", yref="y",
                    x0=coords[0] - r_test, y0=coords[1] - r_test,
                    x1=coords[0] + r_test, y1=coords[1] + r_test,
                    line=dict(color="red", width=2),
                    fillcolor="red",
                    opacity=0.4,
                    layer="above"
                ))
        return shapes

    # Get initial amplitude shapes using the first index.
    init_amplitudes = amplitudes[0]
    init_test_amplitudes = test_amplitudes[0]

    # Create the initial figure.
    fig = go.Figure()

    # Add the sigma heatmap trace (using the first row from fitted_voxels_df).
    row = fitted_voxels_df.iloc[0]
    x0, y0, sigma = row[['x0', 'y0', 'sigma']]
    Z = get_gaussian_heatmap(x0, y0, sigma)
    fig.add_trace(go.Heatmap(
        z=Z, x=x_range, y=y_range,
        colorscale='Reds',
        opacity=0.4,
        showscale=False
    ))

    # Add invisible scatter traces to fix the plot range.
    fig.add_trace(go.Scatter(
        x=[-1, 1], y=[-1, 1],
        mode='markers',
        marker=dict(size=0, opacity=0),
        hoverinfo='skip'
    ))
    fig.add_trace(go.Scatter(
        x=data[:, 0], y=data[:, 1],
        mode='markers',
        marker=dict(size=10, color='rgba(0,0,0,0)'),
        hoverinfo='skip'
    ))

    # Add the initial amplitude shapes.
    init_shapes = get_amplitude_shapes(data, init_amplitudes, init_test_amplitudes, scaling_factor, test_scaling_factor, max_radius)
    fig.update_layout(shapes=init_shapes)

    fig.update_layout(
        title="MDS Plot of Neural Response Similarity",
        xaxis_title="Dimension 1",
        yaxis_title="Dimension 2",
        hovermode='closest',
        xaxis=dict(range=[-1, 1], fixedrange=True, constrain='domain'),
        yaxis=dict(range=[-1, 1], fixedrange=True, scaleanchor="x", scaleratio=1, constrain='domain'),
        width=width, height=height,
        showlegend=False
    )

    # app.layout = html.Div([
    #     dcc.Graph(id='scatter-plot', figure=fig, config={
    #         'doubleClick': 'reset',
    #         'scrollZoom': False,
    #         'displayModeBar': True,
    #         'modeBarButtonsToRemove': ['zoom', 'pan', 'select', 'lasso2d']
    #     }),
    #     html.Div(id='info-panel', style={'textAlign': 'center', 'fontSize': '20px', 'marginBottom': '10px'}),
    #     html.Div([
    #         html.Button("Previous", id="prev-btn", n_clicks=0),
    #         html.Button("Next", id="next-btn", n_clicks=0),
    #     ], style={'textAlign': 'center', 'marginTop': '20px'}),
    #     dcc.Store(id='index-store', data=0)
    # ])

    app.layout = html.Div([
        html.Div([  # New container for side-by-side plots
            dcc.Graph(id='scatter-plot', figure=fig, config={'displayModeBar': False},
                      style={'width': '50%', 'display': 'inline-block'}),
            dcc.Graph(id='amplitude-histogram', 
                      style={'width': '50%', 'display': 'inline-block'}),
        ], style={'display': 'flex'}),
        html.Div(id='info-panel', style={'textAlign': 'center', 'fontSize': '20px'}),
        html.Div([
            html.Button("Previous", id="prev-btn", n_clicks=0),
            html.Button("Next", id="next-btn", n_clicks=0),
        ], style={'textAlign': 'center', 'marginTop': '20px'}),
        dcc.Store(id='index-store', data=0)
    ])

    @app.callback(
        [Output('scatter-plot', 'figure'),
         Output('amplitude-histogram', 'figure'),  # New output
         Output('index-store', 'data'),
         Output('info-panel', 'children')],
        [Input('prev-btn', 'n_clicks'),
         Input('next-btn', 'n_clicks')],
        [State('index-store', 'data')]
    )

    # @app.callback(
    #     [Output('scatter-plot', 'figure'),
    #      Output('index-store', 'data'),
    #      Output('info-panel', 'children')],
    #     [Input('prev-btn', 'n_clicks'),
    #      Input('next-btn', 'n_clicks')],
    #     [State('index-store', 'data')]
    # )
    def update_distribution(prev_clicks, next_clicks, current_index):
        if current_index is None:
            current_index = 0

        ctx = callback_context
        if not ctx.triggered:
            return fig, current_index, "Click a button to change distribution."

        button_id = ctx.triggered[0]['prop_id'].split('.')[0]
        if button_id == "prev-btn":
            new_index = (current_index - 1) % len(fitted_voxels_df.index)
        elif button_id == "next-btn":
            new_index = (current_index + 1) % len(fitted_voxels_df.index)
        else:
            new_index = current_index

        row = fitted_voxels_df.iloc[new_index]
        x0, y0, sigma = row[['x0', 'y0', 'sigma']]
        Z = get_gaussian_heatmap(x0, y0, sigma)

        # Create an updated figure and update the sigma heatmap.
        updated_fig = go.Figure(fig)
        updated_fig.data[0].z = Z

        # Update amplitude shapes with new amplitude values.
        new_amplitudes = amplitudes[new_index]
        new_test_amplitudes = test_amplitudes[new_index]
        new_shapes = get_amplitude_shapes(data, new_amplitudes, new_test_amplitudes,
                                          scaling_factor, test_scaling_factor, max_radius)
        updated_fig.update_layout(shapes=new_shapes)


        fig_hist = go.Figure()
        fig_hist.add_trace(go.Histogram(
            x=new_amplitudes,
            name='Training',
            marker_color='blue',
            opacity=0.7,
            nbinsx=30
        ))
        fig_hist.add_trace(go.Histogram(
            x=new_test_amplitudes,
            name='Test',
            marker_color='red',
            opacity=0.7,
            nbinsx=30
        ))
        
        # Calculate combined range for x-axis
        min_val = min(np.min(new_amplitudes), np.min(new_test_amplitudes))
        max_val = max(np.max(new_amplitudes), np.max(new_test_amplitudes))
        
        fig_hist.update_layout(
            title=f'Amplitude Distribution (Voxel {row.name})',
            barmode='overlay',
            xaxis_title='Amplitude Value',
            yaxis_title='Count',
            xaxis=dict(range=[min_val - 0.1, max_val + 0.1]),
            legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
        )




        # Build the info text with additional amplitude information and color mapping.
        info_text = (
            f"Index: {row.name} | Mean: ({x0:.2f}, {y0:.2f}) | Sigma: {sigma:.2f}<br>"
            f"Training (blue): min={np.min(new_amplitudes):.2f}, max={np.max(new_amplitudes):.2f} | "
            f"Test (red): min={np.min(new_test_amplitudes):.2f}, max={np.max(new_test_amplitudes):.2f}<br>"
            f"(Shapes: circle for positive amplitudes; square for negative)"
        )

        # return updated_fig, new_index, info_text
        return updated_fig, fig_hist, new_index, info_text  # Added fig_hist



    return app


In [ ]:
def create_interactive_circle_scatter_dash(data, amplitudes, test_amplitudes, titles=None,
                                           scaling_factor=0.1, test_scaling_factor=0.1,
                                           max_radius=0.2, width=1500, height=1500):
    """
    Create an interactive scatter plot with amplitude histogram using Dash.
    """
    app = Dash(__name__)

    # Heatmap grid setup
    grid_size = 500
    x_range = np.linspace(-1, 1, grid_size)
    y_range = np.linspace(-1, 1, grid_size)
    X, Y = np.meshgrid(x_range, y_range)

    def get_gaussian_heatmap(x0, y0, sigma):
        Z = np.exp(-((X - x0)**2 + (Y - y0)**2) / (2 * sigma**2))
        return Z / np.max(Z)

    # Helper functions for shapes and visualization components
    def get_amplitude_shapes(data, amps, test_amps, scaling_factor, test_scaling_factor, max_radius):
        shapes = []
        for i, coords in enumerate(data):
            # Training amplitude
            amp = amps[i]
            shape_type = "rect" if amp < 0 else "circle"
            r = min(abs(amp) * scaling_factor, max_radius)
            shapes.append({
                'type': shape_type,
                'xref': "x", 'yref': "y",
                'x0': coords[0] - r, 'y0': coords[1] - r,
                'x1': coords[0] + r, 'y1': coords[1] + r,
                'line': {'color': "blue", 'width': 2},
                'fillcolor': "blue",
                'opacity': 0.8,
                'layer': "above"
            })

            # Test amplitude
            test_amp = test_amps[i]
            test_shape_type = "rect" if test_amp < 0 else "circle"
            r_test = min(abs(test_amp) * test_scaling_factor, max_radius)
            shapes.append({
                'type': test_shape_type,
                'xref': "x", 'yref': "y",
                'x0': coords[0] - r_test, 'y0': coords[1] - r_test,
                'x1': coords[0] + r_test, 'y1': coords[1] + r_test,
                'line': {'color': "red", 'width': 2},
                'fillcolor': "red",
                'opacity': 0.4,
                'layer': "above"
            })
        return shapes

    def create_histogram(train_amps, test_amps, voxel_id):
        fig = go.Figure()
        fig.add_trace(go.Histogram(
            x=train_amps, name='Training', marker_color='blue', opacity=0.7
        ))
        fig.add_trace(go.Histogram(
            x=test_amps, name='Test', marker_color='red', opacity=0.7
        ))
        
        min_val = min(np.min(train_amps), np.min(test_amps))
        max_val = max(np.max(train_amps), np.max(test_amps))
        
        fig.update_layout(
            title=f'Amplitude Distribution (Voxel {voxel_id})',
            barmode='overlay',
            xaxis_title='Amplitude Value',
            yaxis_title='Count',
            xaxis=dict(range=[min_val - 0.1, max_val + 0.1]),
            legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
        )
        return fig

    def generate_info_text(row, train_amps, test_amps):
        """Return information as separate HTML elements for natural line breaks"""
        return html.Div([
            html.Div(f"Index: {row.name} | Mean: ({row.x0:.2f}, {row.y0:.2f}) | Sigma: {row.sigma:.2f}"),
            html.Div([
                "Training (blue): ",
                html.Span(f"min={np.min(train_amps):.2f}", style={'color': 'blue'}),
                ", ",
                html.Span(f"max={np.max(train_amps):.2f}", style={'color': 'blue'})
            ]),
            html.Div([
                "Test (red): ",
                html.Span(f"min={np.min(test_amps):.2f}", style={'color': 'red'}),
                ", ",
                html.Span(f"max={np.max(test_amps):.2f}", style={'color': 'red'})
            ]),
            html.Div("Shapes: ● = positive amplitude, ■ = negative amplitude",
                    style={'marginTop': '10px', 'fontStyle': 'italic'})
        ], style={'lineHeight': '1.8'})

    # Initial figure setup
    fig = go.Figure()
    row = fitted_voxels_df.iloc[0]
    Z = get_gaussian_heatmap(row.x0, row.y0, row.sigma)
    fig.add_trace(go.Heatmap(z=Z, x=x_range, y=y_range, colorscale='Reds', opacity=0.4, showscale=False))
    fig.add_trace(go.Scatter(x=[-1, 1], y=[-1, 1], mode='markers', marker=dict(size=0, opacity=0)))
    fig.add_trace(go.Scatter(x=data[:, 0], y=data[:, 1], mode='markers', marker=dict(size=10, color='rgba(0,0,0,0)')))

    fig.update_layout(
        title="MDS Plot of Neural Response Similarity",
        xaxis_title="Dimension 1",
        yaxis_title="Dimension 2",
        xaxis=dict(range=[-1, 1], fixedrange=True),
        yaxis=dict(range=[-1, 1], fixedrange=True, scaleanchor="x", scaleratio=1),
        width=width//2,  # Split width with histogram
        height=height,
        showlegend=False
    )

    # App layout with side-by-side plots
    app.layout = html.Div([
        # Plot container with side-by-side plots
        html.Div([
            dcc.Graph(id='scatter-plot', figure=fig, 
                    style={'width': '50%', 'display': 'inline-block', 'padding': '0 10px'}),
            dcc.Graph(id='amplitude-histogram',
                    style={'width': '50%', 'display': 'inline-block', 'padding': '0 10px'}),
        ], style={'display': 'flex', 'width': '100%', 'margin': '0 auto'}),
        
        # Information panel with separate elements
        html.Div(id='info-panel', style={'width': '80%', 'margin': '20px auto', 'fontSize': '18px'}),
        
        # Navigation buttons
        html.Div([
            html.Button("Previous", id="prev-btn", 
                    style={'margin': '10px', 'padding': '10px 20px', 'fontSize': '16px'}),
            html.Button("Next", id="next-btn", 
                    style={'margin': '10px', 'padding': '10px 20px', 'fontSize': '16px'}),
        ], style={'textAlign': 'center', 'margin': '20px 0'}),
        
        dcc.Store(id='index-store', data=0)
    ])

    # Callback with complete output handling
    @app.callback(
        [Output('scatter-plot', 'figure'),
         Output('amplitude-histogram', 'figure'),
         Output('index-store', 'data'),
         Output('info-panel', 'children')],
        [Input('prev-btn', 'n_clicks'),
         Input('next-btn', 'n_clicks')],
        [State('index-store', 'data')]
    )
    def update_plots(prev_clicks, next_clicks, current_index):
        ctx = callback_context
        trigger_id = ctx.triggered[0]['prop_id'].split('.')[0] if ctx.triggered else None
        
        # Handle initial load case
        if current_index is None or not trigger_id:
            current_index = 0
            trigger_id = 'next-btn'  # Force initial load

        # Determine new index
        if trigger_id == "prev-btn":
            new_index = (current_index - 1) % len(fitted_voxels_df)
        else:
            new_index = (current_index + 1) % len(fitted_voxels_df)

        # Get current data
        row = fitted_voxels_df.iloc[new_index]
        new_amps = amplitudes[new_index]
        new_test_amps = test_amplitudes[new_index]
        Z = get_gaussian_heatmap(row.x0, row.y0, row.sigma)

        # Update main figure
        updated_fig = go.Figure(fig)
        updated_fig.data[0].z = Z
        updated_fig.update_layout(
            shapes=get_amplitude_shapes(data, new_amps, new_test_amps, 
                                      scaling_factor, test_scaling_factor, max_radius)
        )

        # Create histogram
        hist_fig = create_histogram(new_amps, new_test_amps, row.name)
        
        # Generate info text
        info_text = generate_info_text(row, new_amps, new_test_amps)

        return updated_fig, hist_fig, new_index, info_text

    return app

In [ ]:
from scipy.stats import gaussian_kde


def create_interactive_circle_scatter_dash(data, amplitudes, test_amplitudes, titles=None,
                                           scaling_factor=0.1, test_scaling_factor=0.1,
                                           max_radius=0.2, width=1500, height=800):
    """
    Interactive visualization with single plot selector
    """
    app = Dash(__name__)

    # Heatmap grid setup
    grid_size = 500
    x_range = np.linspace(-1, 1, grid_size)
    y_range = np.linspace(-1, 1, grid_size)
    X, Y = np.meshgrid(x_range, y_range)

    def get_gaussian_heatmap(x0, y0, sigma):
        Z = np.exp(-((X - x0)**2 + (Y - y0)**2) / (2 * sigma**2))
        return Z / np.max(Z)

    def create_density_map(coords, weights):
        """Create density map using absolute weights"""
        abs_weights = np.abs(weights)
        kde = gaussian_kde(coords.T, weights=abs_weights)
        density = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)
        return density / density.max()

    def get_amplitude_shapes(data, amps, test_amps):
        shapes = []
        for i, (x, y) in enumerate(data):
            # Training shape
            amp = amps[i]
            size = min(abs(amp) * scaling_factor, max_radius)
            shapes.append({
                'type': 'rect' if amp < 0 else 'circle',
                'xref': 'x', 'yref': 'y',
                'x0': x - size, 'y0': y - size,
                'x1': x + size, 'y1': y + size,
                'line': {'color': 'blue', 'width': 2},
                'fillcolor': 'blue',
                'opacity': 0.8
            })
            
            # Test shape
            test_amp = test_amps[i]
            test_size = min(abs(test_amp) * test_scaling_factor, max_radius)
            shapes.append({
                'type': 'rect' if test_amp < 0 else 'circle',
                'xref': 'x', 'yref': 'y',
                'x0': x - test_size, 'y0': y - test_size,
                'x1': x + test_size, 'y1': y + test_size,
                'line': {'color': 'red', 'width': 2},
                'fillcolor': 'red',
                'opacity': 0.4
            })
        return shapes

    # App layout
    app.layout = html.Div([
        html.Div([
            dcc.Dropdown(
                id='plot-selector',
                options=[
                    {'label': 'Amplitude Shapes', 'value': 'amplitude'},
                    {'label': 'Amplitude Histogram', 'value': 'histogram'},
                    {'label': 'Training Density Map', 'value': 'density'}
                ],
                value='amplitude',
                style={'width': '400px', 'margin': '10px auto'}
            )
        ], style={'textAlign': 'center'}),
        
        dcc.Graph(id='main-plot', style={'height': '80vh'}),
        
        html.Div(id='info-panel', style={
            'textAlign': 'center',
            'padding': '20px',
            'fontSize': '18px'
        }),
        
        html.Div([
            html.Button("Previous", id="prev-btn", n_clicks=0,
                       style={'margin': '10px', 'padding': '10px 20px'}),
            html.Button("Next", id="next-btn", n_clicks=0,
                       style={'margin': '10px', 'padding': '10px 20px'})
        ], style={'textAlign': 'center'}),
        
        dcc.Store(id='index-store', data=0)
    ])

    @app.callback(
        [Output('main-plot', 'figure'),
         Output('index-store', 'data'),
         Output('info-panel', 'children')],
        [Input('prev-btn', 'n_clicks'),
         Input('next-btn', 'n_clicks'),
         Input('plot-selector', 'value')],
        [State('index-store', 'data')]
    )
    def update_plots(prev_clicks, next_clicks, plot_type, current_index):
        ctx = callback_context
        trigger_id = ctx.triggered[0]['prop_id'].split('.')[0] if ctx.triggered else None
        
        # Handle index changes
        if trigger_id in ['prev-btn', 'next-btn']:
            current_index = (current_index + (-1 if trigger_id == 'prev-btn' else 1)) % len(fitted_voxels_df)
        
        row = fitted_voxels_df.iloc[current_index]
        train_amps = amplitudes[current_index]
        test_amps = test_amplitudes[current_index]
        
        # Create appropriate plot
        if plot_type == 'amplitude':
            fig = go.Figure()
            fig.add_trace(go.Heatmap(
                z=get_gaussian_heatmap(row.x0, row.y0, row.sigma),
                x=x_range, y=y_range,
                colorscale='Reds',
                opacity=0.4,
                showscale=False
            ))
            fig.add_trace(go.Scatter(
                x=data[:, 0], y=data[:, 1], mode='markers',
                marker=dict(size=10, color='rgba(0,0,0,0)')
            ))
            fig.update_layout(shapes=get_amplitude_shapes(data, train_amps, test_amps))
            fig.update_layout(
                title="Amplitude Shapes Visualization",
                xaxis=dict(range=[-1, 1]),
                yaxis=dict(range=[-1, 1], scaleanchor="x")
            )
            
        elif plot_type == 'histogram':
            fig = go.Figure()
            fig.add_trace(go.Histogram(
                x=train_amps, name='Training', marker_color='blue', opacity=0.7, nbinsx=30
            ))
            fig.add_trace(go.Histogram(
                x=test_amps, name='Test', marker_color='red', opacity=0.7, nbinsx=30
            ))
            
            # Calculate dynamic range with 10% padding
            min_val = min(np.min(train_amps), np.min(test_amps))
            max_val = max(np.max(train_amps), np.max(test_amps))
            range_padding = (max_val - min_val) * 0.1
            
            fig.update_layout(
                title='Amplitude Distribution',
                barmode='overlay',
                xaxis_title='Amplitude Value',
                xaxis=dict(range=[min_val - range_padding, max_val + range_padding]),
                yaxis_title='Count',
                showlegend=True
            )
            
        elif plot_type == 'density':
            density = create_density_map(data, train_amps)
            fig = go.Figure()
            fig.add_trace(go.Heatmap(
                z=density,
                x=x_range,
                y=y_range,
                colorscale='Blues',
                opacity=0.8,
                showscale=False
            ))
            fig.update_layout(
                title='Training Amplitudes Density Map',
                xaxis=dict(range=[-1, 1]),
                yaxis=dict(range=[-1, 1], scaleanchor="x")
            )

        # Common layout settings
        fig.update_layout(
            width=width,
            height=height,
            margin=dict(l=20, r=20, t=40, b=20)
        )

        # Information panel
        # Information panel
        info_content = html.Div([
            html.Div(f"Voxel {row.name} | Position: ({row.x0:.2f}, {row.y0:.2f})"),
            html.Div(f"σ: {row.sigma:.2f}"),
            html.Div(f"Slope: {row.slope:.2f} | Solved: {row.solved}"),
            html.Div(f"Training Variance: {row.var_train:.2f} | Test Variance: {row.var_test:.2f}"),
            html.Hr(),
            html.Div([
                html.Div("Explanation of the graphic:"),
                html.Ul([
                    html.Li([
                        "The shapes represent amplitude values. A circle",
                        " indicates a positive value and a square",
                        " indicates a negative value for the training data."
                    ]),
                    html.Li([
                        "For the test data, a circle",
                        " indicates a positive value and a square",
                        " indicates a negative value."
                    ]),
                    html.Li([
                        "The color indicates whether the data is from the ",
                        html.Span("training set", style={'color': 'blue'}),
                        " or the ",
                        html.Span("test set", style={'color': 'red'}),
                        "."
                    ]),
                ])
            ])
        ])


        return fig, current_index, info_content

    return app

In [ ]:


app = create_interactive_circle_scatter_dash(data, betas_masked, betas_test_masked, width=1000, height=1000, max_radius=0.1)
app.run_server(debug=True, port=8050)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Example ROIs (replace with your actual ROI labels)
rois = list(range(1, 7+1))

# Prepare figure
plt.figure(figsize=(8,6))

means = []
stats_data = []

# Compute mean and 95% confidence interval for each ROI
for roi in rois:
    data = pd.read_excel(f"data/gaussian_results/shared/subj01/fitted_voxels_mask_{roi}.xlsx")

    values = data["var_train"].tolist()
    min_val = np.min(values)
    max_val = np.max(values)
    q25 = np.percentile(values, 25)
    q75 = np.percentile(values, 75)
    mean_val = np.mean(values)
    stats_data.append((min_val, q25, q75, max_val, mean_val))
    

# Generate a color gradient across ROIs (you can choose any colormap)
colors = plt.cm.rainbow(np.linspace(0, 1, len(rois)))

for i, roi in enumerate(rois):
    min_val, q25, q75, max_val, mean_val = stats_data[i]
    
    # "Whiskers" from min to max
    plt.plot([i, i], [min_val, max_val], color=colors[i], linewidth=2)
    
    # Thicker line for the interquartile range (IQR) from q25 to q75
    # (Adjust linewidth as you prefer)
    plt.plot([i, i], [q25, q75], color=colors[i], linewidth=6)
    
    # Diamond (♦) marker for the mean
    plt.plot(i, mean_val, 'D', color='black', markersize=8)

# Customize the axes
plt.xticks(range(len(rois)), rois)
plt.xlabel("ROI")
plt.ylabel("Variance explained")
plt.title("Within ROI Model (Min–Max + 25th–75th Percentile)")

# Optionally, set y-axis limits
plt.ylim(-1, 1)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Example ROIs (replace with your actual ROI labels)
rois = list(range(1, 7+1))

# Prepare figure
plt.figure(figsize=(8,6))

means = []
stats_data = []

# Compute mean and 95% confidence interval for each ROI
for roi in rois:
    data = pd.read_excel(f"data/gaussian_results/shared/subj01/fitted_voxels_mask_{roi}.xlsx")

    values = data["var_test"].tolist()
    min_val = np.min(values)
    max_val = np.max(values)
    q25 = np.percentile(values, 25)
    q75 = np.percentile(values, 75)
    mean_val = np.mean(values)
    stats_data.append((min_val, q25, q75, max_val, mean_val))
    

# Generate a color gradient across ROIs (you can choose any colormap)
colors = plt.cm.rainbow(np.linspace(0, 1, len(rois)))

for i, roi in enumerate(rois):
    min_val, q25, q75, max_val, mean_val = stats_data[i]
    
    # "Whiskers" from min to max
    plt.plot([i, i], [min_val, max_val], color=colors[i], linewidth=2)
    
    # Thicker line for the interquartile range (IQR) from q25 to q75
    # (Adjust linewidth as you prefer)
    plt.plot([i, i], [q25, q75], color=colors[i], linewidth=6)
    
    # Diamond (♦) marker for the mean
    plt.plot(i, mean_val, 'D', color='black', markersize=8)

# Customize the axes
plt.xticks(range(len(rois)), rois)
plt.xlabel("ROI")
plt.ylabel("Variance explained")
plt.title("Within ROI Model (Min–Max + 25th–75th Percentile)")

# Optionally, set y-axis limits
plt.ylim(-1, 1)

plt.tight_layout()
plt.show()